In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print("Running Cross-Dataset Generalisation: UNSW-NB15...")

# Load UNSW-NB15
train_unsw = pd.read_csv(r"UNSW_NB15\UNSW_NB15_training-set.csv")
test_unsw  = pd.read_csv(r"UNSW_NB15\UNSW_NB15_testing-set.csv")

print(f"Train shape: {train_unsw.shape}")
print(f"Test shape:  {test_unsw.shape}")
print(f"Columns: {list(train_unsw.columns)}")
print(f"\nUnique labels: {train_unsw['label'].unique()}")

Running Cross-Dataset Generalisation: UNSW-NB15...
Train shape: (175341, 45)
Test shape:  (82332, 45)
Columns: ['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label']

Unique labels: [0 1]


In [2]:
from sklearn.preprocessing import LabelEncoder

print("Preparing UNSW-NB15 data...")

# Drop non-feature columns
drop_cols = ['id', 'attack_cat', 'label']

# Prepare train
X_train_unsw = train_unsw.drop(columns=drop_cols, errors='ignore')
y_train_unsw = train_unsw['label'].astype(int)

# Prepare test
X_test_unsw = test_unsw.drop(columns=drop_cols, errors='ignore')
y_test_unsw = test_unsw['label'].astype(int)

# Handle categorical columns - encode them
cat_cols = X_train_unsw.select_dtypes(include=['object']).columns
print(f"Categorical columns found: {list(cat_cols)}")

for col in cat_cols:
    le = LabelEncoder()
    # Fit on combined to handle unseen categories
    combined = pd.concat([X_train_unsw[col], X_test_unsw[col]], axis=0).astype(str)
    le.fit(combined)
    X_train_unsw[col] = le.transform(X_train_unsw[col].astype(str))
    X_test_unsw[col] = le.transform(X_test_unsw[col].astype(str))

# Fill any remaining missing values
X_train_unsw = X_train_unsw.fillna(0)
X_test_unsw = X_test_unsw.fillna(0)

print(f"Train: {X_train_unsw.shape} | Attacks: {y_train_unsw.sum()} ({y_train_unsw.mean():.2%})")
print(f"Test:  {X_test_unsw.shape}  | Attacks: {y_test_unsw.sum()} ({y_test_unsw.mean():.2%})")

# ── Train RF on UNSW-NB15 ──────────────────────────────────────────
print("\nTraining Random Forest on UNSW-NB15...")
rf_unsw = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_unsw.fit(X_train_unsw, y_train_unsw)
print("Training done!")

# ── Evaluate ───────────────────────────────────────────────────────
preds_unsw = rf_unsw.predict(X_test_unsw)
proba_unsw = rf_unsw.predict_proba(X_test_unsw)[:, 1]

print("\n=== Random Forest on UNSW-NB15 ===")
print(classification_report(y_test_unsw, preds_unsw, target_names=["Benign", "Attack"]))
print("AUC-ROC:", round(roc_auc_score(y_test_unsw, proba_unsw), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_unsw, preds_unsw))

# ── Also train XGBoost for comparison ─────────────────────────────
print("\nTraining XGBoost on UNSW-NB15...")
from xgboost import XGBClassifier
xgb_unsw = XGBClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=len(y_train_unsw[y_train_unsw==0]) / len(y_train_unsw[y_train_unsw==1]),
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
xgb_unsw.fit(X_train_unsw, y_train_unsw)
preds_xgb_unsw = xgb_unsw.predict(X_test_unsw)
proba_xgb_unsw = xgb_unsw.predict_proba(X_test_unsw)[:, 1]

print("\n=== XGBoost on UNSW-NB15 ===")
print(classification_report(y_test_unsw, preds_xgb_unsw, target_names=["Benign", "Attack"]))
print("AUC-ROC:", round(roc_auc_score(y_test_unsw, proba_xgb_unsw), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_unsw, preds_xgb_unsw))

Preparing UNSW-NB15 data...
Categorical columns found: ['proto', 'service', 'state']
Train: (175341, 42) | Attacks: 119341 (68.06%)
Test:  (82332, 42)  | Attacks: 45332 (55.06%)

Training Random Forest on UNSW-NB15...


C:\Users\danje\AppData\Local\Temp\ipykernel_22188\502633302.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train_unsw.select_dtypes(include=['object']).columns


Training done!

=== Random Forest on UNSW-NB15 ===
              precision    recall  f1-score   support

      Benign       0.98      0.73      0.84     37000
      Attack       0.82      0.99      0.90     45332

    accuracy                           0.87     82332
   macro avg       0.90      0.86      0.87     82332
weighted avg       0.89      0.87      0.87     82332

AUC-ROC: 0.9788

Confusion Matrix:
[[27102  9898]
 [  544 44788]]

Training XGBoost on UNSW-NB15...

=== XGBoost on UNSW-NB15 ===
              precision    recall  f1-score   support

      Benign       0.95      0.87      0.91     37000
      Attack       0.90      0.96      0.93     45332

    accuracy                           0.92     82332
   macro avg       0.93      0.92      0.92     82332
weighted avg       0.92      0.92      0.92     82332

AUC-ROC: 0.9846

Confusion Matrix:
[[32323  4677]
 [ 1744 43588]]
